In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Task 7 input/output paths
SHAPE_PATH = PREPROCESSING_DATA_DIR / "demand_hourly_shape_library.parquet"
CALENDAR_HOURLY_PATH = PREPROCESSING_DATA_DIR / "calendar_hourly_2010_2045.parquet"
FUTURE_DAILY_FES_ANCHORED_PATH = OBJ2_DATA_DIR / "future_daily_demand_fes_anchored.parquet"
FUTURE_DEMAND_HOURLY_RAW_PATH = OBJ2_DATA_DIR / "future_demand_hourly_raw_2030_2045.parquet"


In [ ]:
# Convert daily anchored demand to hourly demand

daily = pd.read_parquet(FUTURE_DAILY_FES_ANCHORED_PATH)
daily["date"] = pd.to_datetime(daily["date"])

shape = pd.read_parquet(SHAPE_PATH).copy()
calendar_hourly = pd.read_parquet(CALENDAR_HOURLY_PATH).copy()

calendar_hourly["timestamp_utc"] = pd.to_datetime(calendar_hourly["timestamp_utc"], utc=True)
calendar_hourly["date"] = calendar_hourly["timestamp_utc"].dt.tz_localize(None).dt.floor("D")

calendar_hourly = calendar_hourly[
    (calendar_hourly["timestamp_utc"] >= pd.Timestamp("2030-01-01", tz="UTC")) &
    (calendar_hourly["timestamp_utc"] <= pd.Timestamp("2045-12-31 23:00:00", tz="UTC"))
].copy()


In [ ]:
# Define day type to match convention in shape library

shape_day_types = set(shape["day_type"].astype(str).unique())
print("Shape day_type values:", sorted(shape_day_types))

# Use this only if a future version of the shape library includes holiday profiles.
calendar_hourly["day_type"] = np.select(
    [
        calendar_hourly["is_holiday_gb_any"].astype(int).eq(1),
        calendar_hourly["is_weekend"].astype(int).eq(1),
    ],
    [
        "holiday",
        "weekend",
    ],
    default="weekday",
)

print(calendar_hourly["day_type"].value_counts())

In [ ]:
# Expand daily FES-anchored demand to hourly demand

daily = daily.copy()
shape = shape.copy()
calendar_hourly = calendar_hourly.copy()

daily["date"] = pd.to_datetime(daily["date"]).dt.floor("D")
daily["year"] = daily["year"].astype(int)
daily["fes_scenario"] = daily["fes_scenario"].astype(str)
daily["climate_member"] = daily["climate_member"].astype(str)
daily["anchored_daily_demand_mwh"] = daily["anchored_daily_demand_mwh"].astype("float64")

shape["month"] = shape["month"].astype(int)
shape["hour"] = shape["hour"].astype(int)
shape["day_type"] = shape["day_type"].astype(str)
shape["nd_hour_fraction"] = shape["nd_hour_fraction"].astype("float64")

# Safety checks before joining
assert daily.duplicated(["date", "year", "fes_scenario", "climate_member"]).sum() == 0
assert shape.duplicated(["month", "day_type", "hour"]).sum() == 0

fraction_check = (
    shape
    .groupby(["month", "day_type"], as_index=False)["nd_hour_fraction"]
    .sum()
)

max_fraction_error = (fraction_check["nd_hour_fraction"] - 1.0).abs().max()
print("Max hourly-fraction sum error:", max_fraction_error)

assert max_fraction_error < 1e-10

# Merge daily demand onto hourly calendar
hourly = calendar_hourly.merge(
    daily,
    on=["date", "year"],
    how="inner",
    validate="many_to_many",
)

# Join hourly shape fractions
hourly = hourly.merge(
    shape[["month", "day_type", "hour", "nd_hour_fraction"]],
    on=["month", "day_type", "hour"],
    how="left",
    validate="many_to_one",
)

missing_fraction = hourly["nd_hour_fraction"].isna().sum()

if missing_fraction > 0:
    print(
        hourly.loc[
            hourly["nd_hour_fraction"].isna(),
            ["timestamp_utc", "date", "month", "day_type", "hour",
             "is_weekend", "is_holiday_gb_any"]
        ].drop_duplicates().head(20)
    )

assert missing_fraction == 0, f"Missing hourly fractions: {missing_fraction}"

# Fractions sum to 1 by day, so daily MWh distributed across 24 hours gives hourly MWh.
# Since each row is one hour, hourly MWh is numerically equal to average MW.
hourly["demand_mw"] = (
    hourly["anchored_daily_demand_mwh"] * hourly["nd_hour_fraction"]
).astype("float64")

future_hourly_raw = hourly[
    [
        "timestamp_utc",
        "year",
        "fes_scenario",
        "climate_member",
        "demand_mw",
    ]
].copy()

future_hourly_raw.to_parquet(
    FUTURE_DEMAND_HOURLY_RAW_PATH,
    index=False,
)


In [ ]:
# Reconcile hourly values back to anchored daily demand

hourly_daily_check = (
    hourly
    .groupby(["date", "fes_scenario", "climate_member"], as_index=False)
    .agg(hourly_sum_mwh=("demand_mw", "sum"))
)

daily_check = hourly_daily_check.merge(
    daily[["date", "fes_scenario", "climate_member", "anchored_daily_demand_mwh"]],
    on=["date", "fes_scenario", "climate_member"],
    how="left",
    validate="one_to_one",
)

daily_check["diff_mwh"] = (
    daily_check["hourly_sum_mwh"] - daily_check["anchored_daily_demand_mwh"]
)

print(daily_check["diff_mwh"].abs().describe())
print("Max absolute daily reconciliation difference, MWh:", daily_check["diff_mwh"].abs().max())

assert daily_check["anchored_daily_demand_mwh"].notna().all()
assert daily_check["diff_mwh"].abs().max() < 1e-6

In [ ]:
expected_hours = len(pd.date_range(
    "2030-01-01 00:00:00",
    "2045-12-31 23:00:00",
    freq="h",
    tz="UTC",
))

n_scenarios = future_hourly_raw["fes_scenario"].nunique()
n_members = future_hourly_raw["climate_member"].nunique()

expected_rows = expected_hours * n_scenarios * n_members

print("Expected rows:", expected_rows)
print("Actual rows:", len(future_hourly_raw))

assert len(future_hourly_raw) == expected_rows

dup_keys = future_hourly_raw.duplicated(
    ["timestamp_utc", "fes_scenario", "climate_member"]
).sum()

print("Duplicate hourly keys:", dup_keys)

assert dup_keys == 0
assert future_hourly_raw["demand_mw"].notna().all()
assert (future_hourly_raw["demand_mw"] > 0).all()